In [2]:
import torch
import torch.nn as nn
import pandas as pd

In [ ]:
import torch
import torch.nn as nn

class DownConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DownConvBlock, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        return self.model(x)


class UpConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UpConvBlock, self).__init__()

        self.model = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=2,
                stride=2
            ),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        # Encoder
        self.down1 = DownConvBlock(in_channels, 64)
        self.down2 = DownConvBlock(64, 128)
        self.down3 = DownConvBlock(128, 256)
        self.down4 = DownConvBlock(256, 512)

        # Bottleneck
        self.bottleneck = DownConvBlock(512, 1024)

        # Decoder
        self.up1 = UpConvBlock(1024, 512)
        self.up2 = UpConvBlock(512, 256)
        self.up3 = UpConvBlock(256, 128)
        self.up4 = UpConvBlock(128, 64)

        # Final output layer
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        s1 = self.down1(x)
        p1 = self.pool(s1)

        s2 = self.down2(p1)
        p2 = self.pool(s2)

        s3 = self.down3(p2)
        p3 = self.pool(s3)

        s4 = self.down4(p3)
        p4 = self.pool(s4)

        # Bottleneck
        b = self.bottleneck(p4)

        # Decoder
        d1 = self.up1(b, s4)
        d2 = self.up2(d1, s3)
        d3 = self.up3(d2, s2)
        d4 = self.up4(d3, s1)

        # Output
        out = self.final_conv(d4)

        return out